In [ ]:
import torch
from ultralytics import YOLO

In [ ]:
device = 0 if torch.cuda.is_available() else "cpu"
print(f"Kullanilan cihaz: {'GPU (cuda:0)' if device == 0 else 'CPU'}")
if device == "cpu":
    print("UYARI: GPU bulunamadi, egitim CPU'da yapilacak ve cok yavas olabilir.")

# --- Veri seti yolu ---
# Once depo icindeki dataset/ klasoruna bakar (README'deki kurulum),
# bulamazsa gelistirme makinesindeki konuma duser.
import os

_CANDIDATES = [
    "../dataset/data.yaml",
    "../../../dataset2/yolo26/YOLO.v1i.yolo26/data.yaml",
]
DATA = next((p for p in _CANDIDATES if os.path.exists(p)), _CANDIDATES[0])
print("veri seti:", os.path.abspath(DATA), "->", os.path.exists(DATA))

In [ ]:
# --- YOLOv8n ---
#
# Amac: yolo26n baseline'i literaturdeki diger YOLO surumleriyle kiyaslamak.
# Egitim ayarlari baseline/yolo26.ipynb ile BIREBIR AYNI tutuldu:
#   epochs=500, imgsz=640, batch=16, optimizer=auto (belirtilmedi), plots=True
# Tek degisen sey mimari - kiyas boyle anlamli olur.
#
# Olcek: hepsi 'nano' sinifi (yolo26n 2.50M parametreye karsilik).

model = YOLO("yolov8n.pt")

results = model.train(
    data="../../../dataset2/yolo8/YOLO.v1i.yolov8/data.yaml",
    epochs=500,
    imgsz=640,
    batch=16,
    device=device,
    plots=True,
    patience=100,
    verbose=True,
    name="ds2_yolov8n",
)

In [ ]:
import csv, glob

run = sorted(glob.glob("runs/detect/ds2_yolov8n*/results.csv"))[-1]
rows = list(csv.DictReader(open(run)))
print(run, "| epoch:", len(rows))
for r in rows[:2] + rows[-3:]:
    print("ep %-4s box=%-8s cls=%-8s mAP50=%-7s mAP50-95=%s" % (
        r["epoch"], r["train/box_loss"], r["train/cls_loss"],
        r["metrics/mAP50(B)"], r["metrics/mAP50-95(B)"]))

b = max(rows, key=lambda r: float(r["metrics/mAP50-95(B)"]))
m50, m5095 = float(b["metrics/mAP50(B)"]), float(b["metrics/mAP50-95(B)"])
print()
print("EN IYI epoch %s -> P %.3f R %.3f mAP50 %.4f mAP50-95 %.4f" % (
    b["epoch"], float(b["metrics/precision(B)"]), float(b["metrics/recall(B)"]), m50, m5095))

# yolo26n baseline (ayni valid seti, en iyi epoch)
REF50, REF5095 = 0.8727, 0.6301
print()
print("yolo26n baseline : mAP50 %.4f  mAP50-95 %.4f" % (REF50, REF5095))
print("bu kosu          : mAP50 %.4f  mAP50-95 %.4f" % (m50, m5095))
print("fark             : %+.4f / %+.4f" % (m50 - REF50, m5095 - REF5095))

In [ ]:
from pathlib import Path

best = Path(sorted(glob.glob("runs/detect/ds2_yolov8n*/weights/best.pt"))[-1])
print("test ediliyor:", best)

m = YOLO(str(best))
m.fuse()
m.info(verbose=False, imgsz=640)

res = m.val(
    data="../../../dataset2/yolo8/YOLO.v1i.yolov8/data.yaml",
    split="test",
    imgsz=640,
    device=device,
)
d = res.results_dict
print()
print("=== TEST ===")
for k, lbl in [("metrics/precision(B)", "Precision"),
               ("metrics/recall(B)", "Recall"),
               ("metrics/mAP50(B)", "mAP@0.5"),
               ("metrics/mAP50-95(B)", "mAP@0.5:0.95")]:
    v = d.get(k)
    print(f"{lbl:>14}: {v*100:.2f}%" if v is not None else f"{lbl:>14}: yok")

try:
    for i, c in enumerate(res.names.values()):
        print(f"  {c:>6}: mAP50={res.box.ap50[i]:.4f}  mAP50-95={res.box.ap[i]:.4f}")
except Exception as e:
    print("sinif bazinda metrik alinamadi:", e)

# yolo26n baseline test (279 goruntu / 1023 kutu)
print()
print("yolo26n baseline test: mAP50 88.35%  mAP50-95 63.51%")
print("bu kosu test        : mAP50 %.2f%%  mAP50-95 %.2f%%" % (
    d.get("metrics/mAP50(B)", 0) * 100, d.get("metrics/mAP50-95(B)", 0) * 100))